In [ ]:
# Install Required Dependencies
%pip install --upgrade pip

# Uninstall conflicting packages
%pip uninstall -y langchain_classic langchain-core langchain-openai langchain-community langchain langchain-chroma chromadb beautifulsoup4 python-dotenv PyPDF2 rank_bm25 weaviate-client ragas wikipedia langchain-weaviate langchain-together langchain-experimental tiktoken langgraph langchain-tavily

# PRE-STEP: Install Required Dependencies
%pip install langchain==1.1.0
%pip install langgraph==1.0.4
%pip install langchain-openai==1.1.0
%pip install langchain-community==0.4.1
%pip install langchain-chroma==1.0.0
%pip install chromadb==1.3.5
%pip install python-dotenv==1.1.1
%pip install pydantic==2.12.3
%pip install langmem==0.0.30
%pip install nest-asyncio==1.6.0

In [ ]:
# Cell 1: Establish baseline agent with episodic and semantic memory
# NOTE: We are creating a baseline memory store that will only be used for demonstration purposes in steps 1-5
import os
import sys
from typing import Dict, List, Optional
from pydantic import BaseModel, Field
from baseline_agent import CoALABaselineAgent
from domain_investment.investment_advisor_agent import InvestmentAdvisorAgent
from procedural_memory import ProceduralMemory
from domain_investment.investor_test_scenarios import (
    setup_hierarchy_demo, get_test_cases, get_feedback_rounds
)
from dotenv import load_dotenv
load_dotenv(dotenv_path='env.txt')
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
sys.path.append('domain_investment')

# Initialize the baseline agent (only has episodic and semantic memory)
agent = CoALABaselineAgent(model_name="gpt-4o-mini", temperature=0, persist_directory="./baseline_memory_store")

# TESTING:
# Test the baseline agent
response = agent.process_message(
    "I'm looking to rebalance my portfolio. I'm 35 and have moderate risk tolerance.",
    user_id="demo_investor"
)
print("Baseline Response (without procedural memory):")
print(response)
print(f"\nMemory Stats: {agent.get_memory_stats()}")
print("\nNote: Baseline agent has episodic + semantic memory only")
print("Cells 2-5 will add procedural memory capabilities")

In [ ]:
# Cell 2: Define the structure for procedural memory (NOW GENERIC)
class DomainProcedure(BaseModel):
    """Generic procedure structure for any domain.
    
    Core fields (required for all domains):
    - strategy_pattern: Description of the strategy
    - steps: Ordered list of steps to execute
    - segments: Applicable user segments
    - success_rate: Historical success rate (0.0-1.0)
    - scope: Hierarchy level (global/user/community/task)
    
    Domain-specific data goes in:
    - domain_metrics: Dict for any domain-specific metrics
      (e.g., avg_portfolio_performance for investments,
             quiz_avg for tutoring, etc.)
    """
    # Core fields (domain-agnostic)
    strategy_pattern: str
    steps: List[str]
    segments: List[str] = Field(default_factory=list)
    success_rate: float = 1.0
    usage_count: int = 0
    adaptations: List[Dict] = Field(default_factory=list)
    
    # Segmentation metadata
    scope: str = "global"  
    scope_id: Optional[str] = None
    priority: int = 0
    learned_from_count: int = 1
    
    # Domain-specific metrics stored as flexible dict
    domain_metrics: Dict[str, float] = Field(default_factory=dict)

domain_agent = InvestmentAdvisorAgent()

# TESTING:

# Create a sample procedure for investment domain
sample_procedure = DomainProcedure(
    strategy_pattern="moderate risk portfolio rebalancing",
    steps=[
        "1. Review current allocation and drift from target",
        "2. Assess client's life changes affecting risk tolerance",
        "3. Analyze market conditions and sector rotation opportunities",
        "4. Propose rebalancing with tax-loss harvesting considerations",
        "5. Set up automatic rebalancing schedule"
    ],
    segments=["millennials", "moderate_risk"],  # Generic 'segments' not 'client_segments'
    domain_metrics={
        "avg_portfolio_performance": 8.5,  # Investment-specific metric
        "avg_rebalance_frequency_days": 90  # Another domain metric
    }
)

print("✓ Domain procedure structure created")
print(f"  Strategy: {sample_procedure.strategy_pattern}")
print(f"  Steps: {len(sample_procedure.steps)}")
print(f"  Segments: {', '.join(sample_procedure.segments)}")
print(f"  Domain metrics: {sample_procedure.domain_metrics}")

In [ ]:
# Cell 3: Initialize Procedural Memory System

# Create domain agent (encapsulates all investment-specific logic)
domain_agent = InvestmentAdvisorAgent()

# Create procedural memory with the domain agent
investment_memory = ProceduralMemory(llm=agent.llm, domain_agent=domain_agent)

# TESTING:

print("✓ Procedural memory system initialized")
print(f"  Domain: {domain_agent.__class__.__name__}")

# Verify the system is initialized correctly
stats = investment_memory.get_stats()
print(f"\n📊 Initial state:")
print(f"  Total strategies: {stats['total_strategies']}")
print(f"  By scope: {stats['by_scope']}")
print(f"  Learning history tracking: {len(investment_memory.global_learning_history)} events")

In [ ]:
# Cell 4: Demonstrate Learning from Interactions
# Simulate some successful interactions to learn from
successful_interactions = [
    {
        "query": "I want to rebalance my portfolio",
        "user_id": "user_001",
        "profile": {"age": 35, "risk_tolerance": "moderate"},
        "interaction": {
            "messages": ["User: I want to rebalance", "Assistant: Here's your rebalancing plan..."],
            "success": True,
            "client_satisfaction": 9,
            "returns": 8.5
        }
    },
    {
        "query": "Show me ESG investment options",
        "user_id": "user_002",
        "profile": {"age": 30, "risk_tolerance": "aggressive"},
        "interaction": {
            "messages": ["User: ESG options?", "Assistant: Here are sustainable funds..."],
            "success": True,
            "client_satisfaction": 8,
            "returns": 7.2
        }
    }
]

print("📚 Learning from successful interactions...")
learned_strategies = {}

for data in successful_interactions:
    result = investment_memory.learn_from_interaction(
        query=data["query"],
        interaction_data=data["interaction"],
        user_id=data["user_id"],
        user_profile=data["profile"]
    )
    
    if result.get("learned"):
        for key, value in result.items():
            if "learned" in key:
                learned_strategies[key] = value
                print(f"  ✓ Learned {key}: {value}")

# Check what was learned
stats = investment_memory.get_stats()
print(f"\n📊 After learning:")
print(f"  Total strategies: {stats['total_strategies']}")
print(f"  By scope: {stats['by_scope']}")